### Load the processed data

In [8]:
import pandas as pd

df = pd.read_csv("../data/processed_reviews.csv")

df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,1
1,wonder littl product br br film techniqu unass...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1


In [9]:
df.shape

(49582, 2)

In [10]:
texts = df["review"]
labels = df["sentiment"]

### Split into train and test

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [12]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 39665
Testing samples: 9917


In [15]:
print(X_train.iloc[0])
print("=======================")
print(y_train.iloc[0])

one jehovah wit also work acut care medic facil year seen peopl die hemolyt reaction blood transfus attend numer confer blood born pathogen seen sever patient becom serious ill pathogen induc transfus blood also heard sever jehovah wit told die refus blood 26 year field never actual seen happen leav question realli unreason refus blood transfus commun larg benefit battl issu issu jehovah wit moral one must abstain blood not ambigu statement thank movi allow comment
1


In [16]:
X_train.head()

34595    one jehovah wit also work acut care medic faci...
17243    amongst standard one liner type action film ac...
9409     realti televis crew assign cover small town hi...
21903    funniest scene movi probabl saviour get medal ...
10820    far mainstream film set postcoloni africa one ...
Name: review, dtype: object

In [19]:
X_train.shape

(39665,)

### Build the vocabulary

In [20]:
from collections import Counter

counter = Counter()

for test in X_train:
    tokens = test.split()
    counter.update(tokens)

In [26]:
counter.most_common(10)

[('br', 90347),
 ('movi', 78941),
 ('film', 73004),
 ('not', 49142),
 ('one', 42322),
 ('like', 34975),
 ('time', 23637),
 ('good', 23007),
 ('make', 22915),
 ('see', 22080)]

In [27]:
# Create the vocabulary

word_to_idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in counter:
    word_to_idx[word] = len(word_to_idx)

In [28]:
vocab_size = len(word_to_idx)

print("Vocabulary size:", vocab_size)

Vocabulary size: 124455


### Convert a review into IDs

In [29]:
def encode_text(text):
    tokens = text.split()

    return [
        word_to_idx.get(token, word_to_idx["<UNK>"])
        for token in tokens
    ]

In [31]:
text = X_train.iloc[0]

print(text)
print("======================")
print(encode_text(text))

one jehovah wit also work acut care medic facil year seen peopl die hemolyt reaction blood transfus attend numer confer blood born pathogen seen sever patient becom serious ill pathogen induc transfus blood also heard sever jehovah wit told die refus blood 26 year field never actual seen happen leav question realli unreason refus blood transfus commun larg benefit battl issu issu jehovah wit moral one must abstain blood not ambigu statement thank movi allow comment
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 17, 22, 23, 12, 24, 25, 26, 27, 28, 23, 29, 18, 17, 5, 30, 24, 3, 4, 31, 14, 32, 17, 33, 11, 34, 35, 36, 12, 37, 38, 39, 40, 41, 32, 17, 18, 42, 43, 44, 45, 46, 46, 3, 4, 47, 2, 48, 49, 17, 50, 51, 52, 53, 54, 55, 56]


### Padding the sequences

In [32]:
# Create the padding function

max_length = 100

def pad_sequence(sequence, max_length, pad_idx = 0):
    if len(sequence) < max_length:
        sequence = sequence + [pad_idx] * (max_length - len(sequence))
    else:
        sequence = sequence[:max_length]

    return sequence

In [33]:
X_train_encoded = [encode_text(text) for text in X_train]
X_test_encoded = [encode_text(text) for text in X_test]

In [34]:
X_train_padded = [
    pad_sequence(seq, max_length)
    for seq in X_train_encoded
]

X_test_padded = [
    pad_sequence(seq, max_length)
    for seq in X_test_encoded
]